# Caso 07 — Ampara, seguros: ¿qué canal paga lo que cuesta?
## Día 1 — Primer contacto con los datos

En este primer cuaderno exploramos los **7 ficheros CSV** exportados de los 5 sistemas de Ampara (`canales.csv`, `productos.csv`, `mediadores.csv`, `clientes.csv`, `polizas.csv`, `siniestros.csv` y `campanas.csv`) **sin limpiar ni modificar nada todavía**.

### Objetivos del Día 1:
1. Comprobar dimensiones (`.shape`), primeras filas (`.head()`), tipos de datos (`.info()` / `dtypes`) y nulos (`.isnull().sum()`) de las 7 tablas.
2. Entender el modelo en estrella con la **póliza en el centro** y por qué `polizas.csv` tiene `96.015` filas (mezcla de altas nuevas, renovaciones y duplicados).
3. Auditar las grafías sucias en dimensiones críticas (`canal` en pólizas y campañas, `nombre` vs `ramo` en productos, `tipo` y `estado` en pólizas y siniestros, `fraude`).
4. Confirmar los avisos de Hugo Ferrer:
   - **Duplicados:** por `poliza_id` (reintentos de webhook de Stripe) y por `siniestro_id` (reaperturas).
   - **Tres tipos de vacíos con distinto significado:**
     - **Con significado (conservar):** `mediador_id` vacío en `polizas` (venta digital sin mediador) y `fecha_cierre` vacía en `siniestros` (siniestro abierto en trámite que sigue costando).
     - **Esperado por diseño (ignorar):** `poliza_origen_id` vacío en las pólizas nuevas.
     - **Vacío real (dato desconocido):** `provincia` vacía en `clientes`.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR = Path("data/raw")

tablas = {
    "canales": pd.read_csv(RAW_DIR / "canales.csv"),
    "productos": pd.read_csv(RAW_DIR / "productos.csv"),
    "mediadores": pd.read_csv(RAW_DIR / "mediadores.csv"),
    "campanas": pd.read_csv(RAW_DIR / "campanas.csv"),
    "clientes": pd.read_csv(RAW_DIR / "clientes.csv"),
    "polizas": pd.read_csv(RAW_DIR / "polizas.csv"),
    "siniestros": pd.read_csv(RAW_DIR / "siniestros.csv"),
}

resumen_tablas = pd.DataFrame([
    {
        "tabla": nombre,
        "filas": df.shape[0],
        "columnas": df.shape[1],
        "lista_columnas": ", ".join(df.columns),
    }
    for nombre, df in tablas.items()
])
display(resumen_tablas)


,tabla,filas,columnas,lista_columnas
0,canales,5,3,"canal_id, nombre, tipo"
1,productos,5,4,"producto_id, nombre, ramo, franquicia"
2,mediadores,50,4,"mediador_id, nombre, zona, tipo"
3,campanas,40,7,"campana_id, canal, nombre, fecha_inicio, fecha..."
4,clientes,50000,6,"cliente_id, fecha_nacimiento, edad, provincia,..."
5,polizas,96015,12,"poliza_id, cliente_id, producto_id, canal, med..."
6,siniestros,123366,9,"siniestro_id, poliza_id, fecha_apertura, fecha..."


### 1. Vistazo a las tablas de contexto (`canales`, `productos`, `mediadores`, `campanas`)
Revisamos primero las tablas pequeñas para descubrir los 5 canales canónicos, la trampa del ramo escondido en `productos.nombre`, el vocabulario de canales en `campanas` y la estructura de `mediadores`.


In [2]:
print("=== CANALES (5 canales canónicos) ===")
display(tablas["canales"])

print("\n=== PRODUCTOS (atención a 'nombre' vs 'ramo') ===")
display(tablas["productos"])

print("\n=== MEDIADORES (primeras 5 filas y conteos por zona/tipo) ===")
display(tablas["mediadores"].head())
print("Tipos en mediadores:", tablas["mediadores"]["tipo"].value_counts(dropna=False).to_dict())
print("Zonas en mediadores:", tablas["mediadores"]["zona"].value_counts(dropna=False).to_dict())

print("\n=== CAMPAÑAS (primeras 5 filas y grafías de canal/tipo) ===")
display(tablas["campanas"].head())
print("Grafías de canal en campanas:", tablas["campanas"]["canal"].value_counts(dropna=False).to_dict())
print("Grafías de tipo en campanas:", tablas["campanas"]["tipo"].value_counts(dropna=False).to_dict())


=== CANALES (5 canales canónicos) ===


,canal_id,nombre,tipo
0,1,comparadores,agregador
1,2,directo,digital
2,3,mediadores,Mediado
3,4,banca,banca
4,5,referido,referido



=== PRODUCTOS (atención a 'nombre' vs 'ramo') ===


,producto_id,nombre,ramo,franquicia
0,1,automovil,Auto,"300,00 €"
1,2,vivienda,patrimonial,"150,00 €"
2,3,Salud,Personal,"0,00 €"
3,4,Vida,Personal,NaN
4,5,mascota,Patrimonial,"90,00 €"



=== MEDIADORES (primeras 5 filas y conteos por zona/tipo) ===


,mediador_id,nombre,zona,tipo
0,1,Correduría Iberia,norte,agente
1,2,Seguros Atlántico,Centro,Corredor
2,3,Mediación Delta,Levante,Corredor
3,4,Grupo Aseguranza,sur,Agente
4,5,Vega Correduría,Norte,corredor


Tipos en mediadores: {'Corredor': 14, 'corredor': 13, 'agente': 11, 'agencia': 6, 'Agente': 3, 'Agencia': 3}
Zonas en mediadores: {'norte': 9, 'Levante': 7, 'centro': 7, 'Sur': 7, 'Centro': 6, 'sur': 5, 'levante': 5, 'Norte': 4}

=== CAMPAÑAS (primeras 5 filas y grafías de canal/tipo) ===


,campana_id,canal,nombre,fecha_inicio,fecha_fin,coste,tipo
0,9001,agregador,Fee Rastreator,23/08/2023,31 Oct 2023,"36.062,64 €",Fee_comparador
1,9002,comparadores,Acuerdo Agregador A,2023-09-25,2024-02-11,"42.551,40 €",fee
2,9003,agregador,Acuerdo Agregador B,18/05/2023,2023-10-19,"48.790,60 €",fee
3,9004,Rastreator,Puja Comparador Q1,2023-11-15,23/02/2024,"37.590,92 €",fee
4,9005,comparadores,Puja Comparador Q2,02/12/2023,2024-01-14,"25.011,76 €",Fee_comparador


Grafías de canal en campanas: {'comparadores': 6, 'comision': 5, 'ads': 4, 'acuerdo': 4, 'recompensa': 4, 'agregador': 3, 'directo': 3, 'mediadores': 3, 'banca': 3, 'paid': 2, 'referido': 2, 'Rastreator': 1}
Grafías de tipo en campanas: {'fee': 5, 'ads': 5, 'comision': 5, 'Acuerdo_banca': 4, 'Fee_comparador': 3, 'Publicidad': 3, 'Comisión': 3, 'referral': 3, 'fee_comparador': 2, 'acuerdo': 2, 'Recompensa': 2, 'publicidad': 1, 'acuerdo_banca': 1, 'recompensa': 1}


### 2. Radiografía de las tablas principales (`clientes`, `polizas`, `siniestros`)
Revisamos `.head()`, tipos de datos y recuento de vacíos (`.isnull().sum()`) en las tres tablas grandes.


In [3]:
for nombre in ["clientes", "polizas", "siniestros"]:
    df = tablas[nombre]
    print(f"================ {nombre.upper()} ({df.shape[0]:,} filas x {df.shape[1]} cols) ================")
    display(df.head(4))
    info_df = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "nulos": df.isnull().sum(),
        "pct_nulos": (df.isnull().mean() * 100).round(2),
        "n_unicos": df.nunique(dropna=True),
    })
    display(info_df)


================ CLIENTES (50,000 filas x 6 cols) ================


,cliente_id,fecha_nacimiento,edad,provincia,sexo,fecha_alta
0,1,02/06/1990,34,VLC,mujer,2022-06-23
1,2,17 Jul 1979,45,MLG,hombre,2021-03-05
2,3,19/06/1981,43,Madrid,mujer,27/11/2020
3,4,1986-03-22,38,MAD,Mujer,2020-03-04


,dtype,nulos,pct_nulos,n_unicos
cliente_id,int64,0,0.0,50000
fecha_nacimiento,str,0,0.0,27211
edad,int64,0,0.0,70
provincia,str,1500,3.0,24
sexo,str,1000,2.0,7
fecha_alta,str,0,0.0,2735


================ POLIZAS (96,015 filas x 12 cols) ================


,poliza_id,cliente_id,producto_id,canal,mediador_id,prima_anual,descuento_pct,fecha_efecto,fecha_vencimiento,tipo,poliza_origen_id,estado
0,813523,26994,1,web,NaN,"539,34 €",10 %,19/09/2024,2025-09-19,renovacion,129883.0,vigente
1,125742,7937,1,directo,NaN,"605,19 €",10 %,29/09/2023,29/09/2024,Nueva,NaN,Vencida
2,115795,39731,2,comparadores,NaN,"418,51 €",0 %,2022-08-29,2023-08-29,nueva,NaN,Vigente
3,821506,24472,1,agente,1.0,"792,47 €",0 %,2024-04-28,2025-04-28,renovacion,141649.0,Vigente


,dtype,nulos,pct_nulos,n_unicos
poliza_id,int64,0,0.00,95415
cliente_id,int64,0,0.00,50000
producto_id,int64,0,0.00,5
canal,str,0,0.00,22
mediador_id,float64,66202,68.95,50
prima_anual,str,0,0.00,48073
descuento_pct,str,16648,17.34,6
fecha_efecto,str,0,0.00,2753
fecha_vencimiento,str,0,0.00,2742
tipo,str,0,0.00,7


================ SINIESTROS (123,366 filas x 9 cols) ================


,siniestro_id,poliza_id,fecha_apertura,fecha_cierre,importe,tipo,estado,fraude,dias_gestion
0,531998,114415,15/08/2023,2023-10-09,"87,95 €",Consulta,cerrado,no,55.0
1,606493,151706,25/07/2023,2023-12-05,"133,09 €",incendio,Pagado,0,133.0
2,579468,136943,22/09/2023,NaN,"30,27 €",daños por agua,Abierto,false,NaN
3,515912,107238,15 Apr 2024,NaN,"26,10 €",cirugia,abierto,no,NaN


,dtype,nulos,pct_nulos,n_unicos
siniestro_id,int64,0,0.00,122509
poliza_id,int64,0,0.00,43269
fecha_apertura,str,0,0.00,2714
fecha_cierre,str,20568,16.67,2601
importe,str,0,0.00,40999
tipo,str,0,0.00,20
estado,str,0,0.00,8
fraude,str,0,0.00,8
dias_gestion,float64,20568,16.67,213


### 3. Auditoría de grafías sucias en `polizas` y `siniestros`
Comprobamos cuántas grafías distintas existen para `canal`, `tipo` y `estado` en `polizas`, así como `tipo`, `estado` y `fraude` en `siniestros`.


In [4]:
polizas = tablas["polizas"]
siniestros = tablas["siniestros"]

print(f"Grafías distintas de 'canal' en polizas ({polizas['canal'].nunique()} grafías):")
print(polizas["canal"].value_counts(dropna=False))

print("\nGrafías de 'tipo' en polizas:")
print(polizas["tipo"].value_counts(dropna=False))

print("\nGrafías de 'estado' en polizas:")
print(polizas["estado"].value_counts(dropna=False))

print("\nGrafías de 'estado' en siniestros:")
print(siniestros["estado"].value_counts(dropna=False))

print("\nGrafías de 'fraude' en siniestros:")
print(siniestros["fraude"].value_counts(dropna=False))


Grafías distintas de 'canal' en polizas (22 grafías):
canal
comparadores     13720
directo           9024
mediadores        8605
Comparadores      8145
banca             5495
Directo           4666
comparador        4636
Mediador          4593
Rastreator        4153
referido          3785
web               3505
corredor          3494
agregador         3435
Banca             2976
agente            2639
app               2598
bancaseguros      2210
Referido          2113
online            1716
banco             1711
recomendacion     1584
amigo             1212
Name: count, dtype: int64

Grafías de 'tipo' en polizas:


tipo
nueva         30327
Nueva         18078
renovacion    15898
alta          12210
Renovación     8751
renov          6420
renovada       4331
Name: count, dtype: int64

Grafías de 'estado' en polizas:
estado
vigente      38031
Vigente      22762
activa       15263
Vencida       5859
vencida       5780
anulada       4211
cancelada     2518
baja          1591
Name: count, dtype: int64

Grafías de 'estado' en siniestros:
estado
pagado        49315
Pagado        29856
cerrado       19625
abierto       10286
Abierto        6219
en tramite     4063
rechazado      2526
denegado       1476
Name: count, dtype: int64

Grafías de 'fraude' en siniestros:
fraude
0        29244
No       29124
no       29105
false    29063
1         1757
si        1713
Sí        1704
true      1656
Name: count, dtype: int64


### 4. Confirmación de los avisos de Hugo: Duplicados y los 3 tipos de vacíos
Verificamos con exactitud:
1. Cuántas filas duplicadas por identificador (`poliza_id` y `siniestro_id`) traen los ficheros.
2. Cuánto pesan los **dos vacíos con significado** (`mediador_id` vacío = venta digital, `fecha_cierre` vacía = siniestro abierto), el **vacío esperado** (`poliza_origen_id` vacío en pólizas nuevas) y el **vacío real** (`provincia` vacía en clientes).


In [5]:
clientes = tablas["clientes"]

dup_polizas = polizas.duplicated(subset=["poliza_id"]).sum()
dup_siniestros = siniestros.duplicated(subset=["siniestro_id"]).sum()

print("=== 1. DUPLICADOS POR ID ===")
print(f"- Pólizas totales: {len(polizas):,} | IDs únicos: {polizas['poliza_id'].nunique():,} | Duplicados por poliza_id: {dup_polizas:,}")
print(f"- Siniestros totales: {len(siniestros):,} | IDs únicos: {siniestros['siniestro_id'].nunique():,} | Duplicados por siniestro_id: {dup_siniestros:,}")

print("\n=== 2. TRES TIPOS DE VACÍOS ===")
n_sin_mediador = polizas["mediador_id"].isnull().sum()
pct_sin_mediador = polizas["mediador_id"].isnull().mean() * 100

n_sin_cierre = siniestros["fecha_cierre"].isnull().sum()
pct_sin_cierre = siniestros["fecha_cierre"].isnull().mean() * 100

n_sin_origen = polizas["poliza_origen_id"].isnull().sum()
pct_sin_origen = polizas["poliza_origen_id"].isnull().mean() * 100

n_sin_descuento = polizas["descuento_pct"].isnull().sum()
pct_sin_descuento = polizas["descuento_pct"].isnull().mean() * 100

n_sin_provincia = clientes["provincia"].isnull().sum()
pct_sin_provincia = clientes["provincia"].isnull().mean() * 100

resumen_vacios = pd.DataFrame([
    {
        "tabla.columna": "polizas.mediador_id",
        "tipo_vacio": "CON SIGNIFICADO (conservar)",
        "nulos": n_sin_mediador,
        "pct": round(pct_sin_mediador, 2),
        "interpretacion": "Venta digital (comparadores, directo/web/app) sin mediador -> usar LEFT JOIN",
    },
    {
        "tabla.columna": "siniestros.fecha_cierre",
        "tipo_vacio": "CON SIGNIFICADO (conservar)",
        "nulos": n_sin_cierre,
        "pct": round(pct_sin_cierre, 2),
        "interpretacion": "Siniestro abierto en trámite (reserva / importe incurrido que sigue costando)",
    },
    {
        "tabla.columna": "polizas.poliza_origen_id",
        "tipo_vacio": "ESPERADO POR DISEÑO (ignorar)",
        "nulos": n_sin_origen,
        "pct": round(pct_sin_origen, 2),
        "interpretacion": "Pólizas nuevas (solo las renovaciones apuntan a una póliza original)",
    },
    {
        "tabla.columna": "polizas.descuento_pct",
        "tipo_vacio": "ESPERADO POR DISEÑO (0%)",
        "nulos": n_sin_descuento,
        "pct": round(pct_sin_descuento, 2),
        "interpretacion": "Póliza contratada sin descuento comercial",
    },
    {
        "tabla.columna": "clientes.provincia",
        "tipo_vacio": "REAL / FALTA REGISTRO",
        "nulos": n_sin_provincia,
        "pct": round(pct_sin_provincia, 2),
        "interpretacion": "Dato desconocido (no se registró al dar de alta al cliente)",
    },
])
display(resumen_vacios)


=== 1. DUPLICADOS POR ID ===


- Pólizas totales: 96,015 | IDs únicos: 95,415 | Duplicados por poliza_id: 600
- Siniestros totales: 123,366 | IDs únicos: 122,509 | Duplicados por siniestro_id: 857

=== 2. TRES TIPOS DE VACÍOS ===


,tabla.columna,tipo_vacio,nulos,pct,interpretacion
0,polizas.mediador_id,CON SIGNIFICADO (conservar),66202,68.95,"Venta digital (comparadores, directo/web/app) ..."
1,siniestros.fecha_cierre,CON SIGNIFICADO (conservar),20568,16.67,Siniestro abierto en trámite (reserva / import...
2,polizas.poliza_origen_id,ESPERADO POR DISEÑO (ignorar),60615,63.13,Pólizas nuevas (solo las renovaciones apuntan ...
3,polizas.descuento_pct,ESPERADO POR DISEÑO (0%),16648,17.34,Póliza contratada sin descuento comercial
4,clientes.provincia,REAL / FALTA REGISTRO,1500,3.00,Dato desconocido (no se registró al dar de alt...
